In [0]:
loan_info=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/loans/")
loan_info.createOrReplaceTempView("loan_info")


In [0]:
%sql
select * from loan_info

In [0]:
payment_transaction=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/payment/")
payment_transaction.createOrReplaceTempView("payment_transaction")

In [0]:
%sql
select * from payment_transaction

In [0]:
df_transaction=spark.sql(f"""with extracted_components as (
    select 
        pt.amount as amount,
        pt.loan_id as loan_id,
        pt.metadata as metadata,
        pt.payment_id as payment_id,
        pt.payment_method as payment_method,
        pt.timestamp as timestamp,
        li.origination_date as origination_date,
        li.customer_id as customer_id,
        li.product_type as product_type,
        li.principal_amount as principal_amount,
        EXTRACT(YEAR FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_year,
        EXTRACT(MONTH FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_month,
        EXTRACT(DAY FROM CAST(li.origination_date AS DATE)) AS orig_day
    from payment_transaction pt
    inner join loan_info li on pt.loan_id = li.loan_id
    where li.status in ('active', 'default')
),

rollover_logic as (
    select
        amount,
        loan_id,
        metadata,
        payment_id,
        payment_method,
        timestamp,
        origination_date,
        customer_id,
        product_type,
        target_year,
        target_month,
        orig_day,
        principal_amount,
        -- Apply the calendar logic explicitly 
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 1
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN 1
            ELSE orig_day 
        END AS final_day,
        
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 3 
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN target_month + 1 
            ELSE target_month
        END AS final_month
    from extracted_components
)

select 
    amount,
    loan_id,
    metadata,
    payment_id,
    payment_method,
    timestamp,
    customer_id,
    product_type,
    origination_date,
    principal_amount,
    make_date(target_year, final_month, final_day) AS due_date 
from rollover_logic""")

df_transaction.createOrReplaceTempView("df_transaction")

In [0]:
df_transaction.display()
df_transaction.printSchema()

In [0]:
from pyspark.sql.functions import col, max, min, current_date, datediff, to_date, sum, count, round, when

# 1. Standardize dates
df_audited = df_transaction.withColumn("payment_date", to_date(col("timestamp")))

# 2. Aggregate Freshness and Completeness metrics per Payment Method (Source Channel)
# --- CORRECTED AGGREGATION BLOCK ---
df_data_quality = df_audited.groupBy("payment_method").agg(
    count("payment_id").alias("total_records_received"),
    
    # --- FRESHNESS ---
    max("payment_date").alias("latest_record_date"),
    datediff(current_date(), max("payment_date")).alias("days_of_staleness"),
    
    # --- COMPLETENESS ---
    # FIXED: Only use .isNull() for the STRUCT data type
    sum(when(col("metadata").isNull(), 1).otherwise(0)).alias("records_with_missing_metadata"),
    
    sum(when(col("timestamp").isNull(), 1).otherwise(0)).alias("unexecuted_payments")
)

# 3. Add completeness percentages for the final report
df_dq_report = df_data_quality.withColumn(
    "metadata_completeness_pct",
    round(((col("total_records_received") - col("records_with_missing_metadata")) / col("total_records_received")) * 100, 2)
)

df_dq_report.select(
    "payment_method",
    "total_records_received",
    "latest_record_date",
    "days_of_staleness",
    "metadata_completeness_pct",
    "unexecuted_payments"
).orderBy("days_of_staleness").display()

In [0]:
df_dq_report.write.mode('overwrite').parquet("abfss://gold@deassessment.dfs.core.windows.net/data_freshness_completeness")